In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("glenmore_traffic.csv")
df.info()

In [ ]:
# convert the study date to datetime for plotting
df["STUDY_DATE"] = pd.to_datetime(df["STUDY_DATE"], format=r"%m/%d/%Y %I:%M:%S %p")

In [ ]:
for dir in ["E", "W"]:
    this_dir = df[df["DIRECTION"] == dir]
    plt.plot(this_dir["STUDY_DATE"], this_dir["VOLUME"], label=dir)

plt.legend()
# zoom in so we can see a bit more detail
# plt.xlim(pd.to_datetime(["2023-01-01", "2023-01-2"]))

In [ ]:
# autocorrelation?
t = df["VOLUME"][df["DIRECTION"] == "W"][1:]
t_minus_1 = df["VOLUME"][df["DIRECTION"] == "W"][:-1]
plt.scatter(t, t_minus_1, alpha=0.3)
np.corrcoef(t, t_minus_1)

In [ ]:
# pull the date/month/time info out of the datetime object
def extract_date_features(df):
    return df[["STUDY_DATE"]].astype(int)
    
    # df["DAY_OF_WEEK"] = df["STUDY_DATE"].dt.day_of_week
    # df["DAY_OF_MONTH"] = df["STUDY_DATE"].dt.day
    # df["MONTH"] = df["STUDY_DATE"].dt.month
    # df["TIME_HOUR"] = df["STUDY_DATE"].dt.hour
    # df["TIME_MINUTE"] = df["STUDY_DATE"].dt.minute

    return df.drop(columns="STUDY_DATE")


In [ ]:
plt.scatter(extract_date_features(df), df["VOLUME"])

In [ ]:
# try a super basic model
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor

# hold back the last 20% for testing
threshold = int(0.8 * len(df))
train_val, test = df[:threshold], df[threshold:]

# BIG MISTAKE! Can't randomly split time series data
train, val = train_test_split(train_val, test_size=0.2, random_state=1234)

pipeline = make_pipeline (
        make_column_transformer(
            (OneHotEncoder(sparse_output=False, drop="if_binary"), ["DIRECTION"]),
            (FunctionTransformer(extract_date_features), ["STUDY_DATE"])
        ),
        DecisionTreeRegressor(),
)

pipeline.fit(train, train["VOLUME"])

In [ ]:
def predict_plot(df):
    y_est = pipeline.predict(df)
    plt.scatter(df["VOLUME"], y_est, alpha=0.3)
    plt.text(0.1 * max(y_est), 0.8*max(y_est), f"MAE = {np.abs(df["VOLUME"] - y_est).mean():.1f}")

predict_plot(train)

In [ ]:
# what about validation data?
predict_plot(val)

## Code for plots

In [ ]:
# encoding hours
hours = np.hstack([np.arange(24)]*7)
plt.plot(hours)
plt.xlabel("Sample")
plt.ylabel("Hour of the day")
plt.tight_layout()
plt.savefig("../../img/12-hours.png")